In [12]:

import os
import psutil

print("=== COLAB HARDWARE ===")

print("Logical CPUs:", psutil.cpu_count(logical=True))
print("Physical CPUs:", psutil.cpu_count(logical=False))

print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

print("Total RAM:",
      round(psutil.virtual_memory().total / (1024**3), 2), "GB")

print("CPU Usage:", psutil.cpu_percent(interval=1), "%")


=== COLAB HARDWARE ===
Logical CPUs: 2
Physical CPUs: 1
Available RAM: 11.31 GB
Total RAM: 12.67 GB
CPU Usage: 2.0 %


In [ ]:
-----------------------------------------------------------------------------------------------------------------

In [2]:
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 57.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit-aer]


In [3]:
!pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.8/551.8 kB 15.6 MB/s  0:00:00
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [qiskit-ibm-runtime]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 3.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 3.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<

In [18]:
from math import pi as π
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_state_city, plot_state_qsphere
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import RGQFTMultiplier
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_distribution
from qiskit_ibm_runtime import SamplerV2, EstimatorV2, QiskitRuntimeService
import psutil
import time
# Record the start time
start_time = time.time()

In [19]:

# ==========================================
# PART 2: MINI-AES + INVERSE + ORACLE
# ==========================================

def Plaintext(circuit, source, target, length):
    for i in range(length):
        if (source >> i) & 1:
            circuit.x(target[i])


# Only the Low Byte is unknown.
# K = 0x00XX
def Input_key(qc, key_reg):
    for i in range(8):
        qc.h(key_reg[i])


def Check_Cipher(qc, cipher_text, target, length):
    for i in range(length):
        if ((cipher_text >> i) & 1) == 0:
            qc.x(target[i])


def XOR(qc, a, b, start_a, start_b, length):
    for i in range(length):
        qc.cx(a[start_a + i], b[start_b + i])


# ------------------------------------------
# SBOX
# ------------------------------------------

def SBOX(qc, qubits):
    a, b, c, d = qubits

    qc.swap(c, d)
    qc.cx(d, a)
    qc.cx(c, d)
    qc.ccx(d, a, b)
    qc.x(d)
    qc.cx(b, c)
    qc.ccx(c, b, a)
    qc.mcx([a, b, d], c)
    qc.x(c)
    qc.mcx([a, b, c], d)
    qc.cx(a, b)
    qc.cx(b, d)
    qc.x(a)
    qc.swap(a, b)
    qc.swap(d, a)

    return qc


def inverse_sbox(qc, qubits):
    a, b, c, d = qubits

    qc.swap(d, a)
    qc.swap(a, b)
    qc.x(a)
    qc.cx(b, d)
    qc.cx(a, b)
    qc.mcx([a, b, c], d)
    qc.x(c)
    qc.mcx([a, b, d], c)
    qc.ccx(c, b, a)
    qc.cx(b, c)
    qc.x(d)
    qc.ccx(d, a, b)
    qc.cx(c, d)
    qc.cx(d, a)
    qc.swap(c, d)

    return qc


# ------------------------------------------
# SHIFT ROWS
# These functions use absolute indices.
# Therefore, state_reg must be the first
# register in the QuantumCircuit.
# ------------------------------------------

def Row_Shift(qc):
    qc.swap(4, 12)
    qc.swap(5, 13)
    qc.swap(6, 14)
    qc.swap(7, 15)


def Inverse_Row_Shift(qc):
    qc.swap(7, 15)
    qc.swap(6, 14)
    qc.swap(5, 13)
    qc.swap(4, 12)


# ------------------------------------------
# MIX COLUMNS
# ------------------------------------------

def mix_column(qc, qubits_1):
    a, b, c, d = qubits_1

    qc.swap(a, d+4)
    qc.swap(b, d+3)
    qc.swap(c, d+2)
    qc.swap(d, d+1)

    qc.cx(d, a)
    qc.cx(d+1, a)
    qc.cx(d+2, a)
    qc.cx(d+4, a)

    qc.cx(d+2, b)
    qc.cx(d+3, b)
    qc.cx(d+3, c)
    qc.cx(d+4, c)

    qc.cx(d+1, d)
    qc.cx(d+4, d)

    qc.cx(c, d+4)
    qc.cx(b, d+4)
    qc.cx(b, d+3)
    qc.cx(a, d+3)

    qc.cx(d+1, d+2)
    qc.cx(a, d+2)
    qc.cx(d, d+1)
    qc.cx(c, d+1)

    qc.swap(a, c)
    qc.swap(d+2, d+4)


def Inverse_mix_column(qc, qubits_1):
    a, b, c, d = qubits_1

    qc.swap(d+2, d+4)
    qc.swap(a, c)

    qc.cx(c, d+1)
    qc.cx(d, d+1)
    qc.cx(a, d+2)
    qc.cx(d+1, d+2)

    qc.cx(a, d+3)
    qc.cx(b, d+3)
    qc.cx(b, d+4)
    qc.cx(c, d+4)

    qc.cx(d+4, d)
    qc.cx(d+1, d)
    qc.cx(d+4, c)
    qc.cx(d+3, c)

    qc.cx(d+3, b)
    qc.cx(d+2, b)
    qc.cx(d+4, a)
    qc.cx(d+2, a)

    qc.cx(d+1, a)
    qc.cx(d, a)

    qc.swap(d, d+1)
    qc.swap(c, d+2)
    qc.swap(b, d+3)
    qc.swap(a, d+4)


# ==========================================
# MINI AES ENCRYPTION
# ==========================================

def Mini_AES(qc, key_reg, state_reg, length):

    # Initial AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()

    # Round 1: SubBytes
    for i in range(0, 16, 4):
        SBOX(qc, (
            state_reg[i],
            state_reg[i+1],
            state_reg[i+2],
            state_reg[i+3]
        ))

    # Round 1: Key Schedule
    SBOX(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    XOR(qc, key_reg, key_reg, 12, 0, 4)
    qc.x(key_reg[3])

    inverse_sbox(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.barrier()

    XOR(qc, key_reg, key_reg, 0, 4, 4)
    XOR(qc, key_reg, key_reg, 4, 8, 4)
    XOR(qc, key_reg, key_reg, 8, 12, 4)

    qc.barrier()

    # ShiftRows
    Row_Shift(qc)
    qc.barrier()

    # MixColumns
    mix_column(qc, (0, 1, 2, 3))
    mix_column(qc, (8, 9, 10, 11))

    qc.barrier()

    # AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()

    # Round 2: SubBytes
    for i in range(0, 16, 4):
        SBOX(qc, (
            state_reg[i],
            state_reg[i+1],
            state_reg[i+2],
            state_reg[i+3]
        ))

    # Round 2: Key Schedule
    SBOX(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    XOR(qc, key_reg, key_reg, 12, 0, 4)
    qc.x(key_reg[2])

    inverse_sbox(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.barrier()

    XOR(qc, key_reg, key_reg, 0, 4, 4)
    XOR(qc, key_reg, key_reg, 4, 8, 4)
    XOR(qc, key_reg, key_reg, 8, 12, 4)

    qc.barrier()

    # ShiftRows
    Row_Shift(qc)
    qc.barrier()

    # Final AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()


# ==========================================
# INVERSE MINI AES
# ==========================================

def Inverse_Mini_AES(qc, key_reg, state_reg, length):

    # Undo final AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()

    # Undo Round 2 ShiftRows
    Inverse_Row_Shift(qc)
    qc.barrier()

    # Undo Round 2 Key Schedule
    XOR(qc, key_reg, key_reg, 8, 12, 4)
    XOR(qc, key_reg, key_reg, 4, 8, 4)
    XOR(qc, key_reg, key_reg, 0, 4, 4)

    SBOX(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.x(key_reg[2])

    XOR(qc, key_reg, key_reg, 12, 0, 4)

    inverse_sbox(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.barrier()

    # Undo Round 2 SubBytes
    for i in reversed(range(0, 16, 4)):
        inverse_sbox(qc, (
            state_reg[i],
            state_reg[i+1],
            state_reg[i+2],
            state_reg[i+3]
        ))

    qc.barrier()

    # Undo Round 1 AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()

    # Undo MixColumns
    Inverse_mix_column(qc, (0, 1, 2, 3))
    Inverse_mix_column(qc, (8, 9, 10, 11))
    qc.barrier()

    # Undo Round 1 ShiftRows
    Inverse_Row_Shift(qc)
    qc.barrier()

    # Undo Round 1 Key Schedule
    XOR(qc, key_reg, key_reg, 8, 12, 4)
    XOR(qc, key_reg, key_reg, 4, 8, 4)
    XOR(qc, key_reg, key_reg, 0, 4, 4)

    SBOX(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.x(key_reg[3])

    XOR(qc, key_reg, key_reg, 12, 0, 4)

    inverse_sbox(qc, (
        key_reg[12],
        key_reg[13],
        key_reg[14],
        key_reg[15]
    ))

    qc.barrier()

    # Undo Round 1 SubBytes
    for i in reversed(range(0, 16, 4)):
        inverse_sbox(qc, (
            state_reg[i],
            state_reg[i+1],
            state_reg[i+2],
            state_reg[i+3]
        ))

    qc.barrier()

    # Undo Initial AddRoundKey
    XOR(qc, key_reg, state_reg, 0, 0, length)
    qc.barrier()


# ==========================================
# GROVER ORACLE
# ==========================================

def my_oracle(qc, state_reg, key_reg, length, cipher_text):

    # Forward encryption
    Mini_AES(qc, key_reg, state_reg, length)
    qc.barrier()

    # Compare with target ciphertext
    Check_Cipher(qc, cipher_text, state_reg, length)

    # Phase marking
    qc.h(state_reg[15])

    control_qubits = list(state_reg[:15])
    target_qubit = state_reg[15]

    qc.mcx(control_qubits, target_qubit)

    qc.h(state_reg[15])

    # Undo ciphertext comparison
    Check_Cipher(qc, cipher_text, state_reg, length)
    qc.barrier()

    # Reverse encryption
    Inverse_Mini_AES(qc, key_reg, state_reg, length)
    qc.barrier()


# ==========================================
# GROVER DIFFUSER
# ==========================================

def diffuser(nqubits):

    qc = QuantumCircuit(nqubits)

    for qubit in range(nqubits):
        qc.h(qubit)

    for qubit in range(nqubits):
        qc.x(qubit)

    qc.h(nqubits - 1)

    qc.mcx(
        list(range(nqubits - 1)),
        nqubits - 1
    )

    qc.h(nqubits - 1)

    for qubit in range(nqubits):
        qc.x(qubit)

    for qubit in range(nqubits):
        qc.h(qubit)

    U_s = qc.to_gate()
    U_s.name = "Diffuser"

    return U_s


# ==========================================
# MEMORY USAGE
# ==========================================

def get_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / (1024**2)


In [21]:

# ==========================================
# PART 3: QUANTUM CIRCUIT + GROVER SEARCH
# ==========================================

# Known plaintext and target ciphertext
plaintext = 0xBAD0
cipher_text = 0x79AE

length = 16

# ------------------------------------------
# Quantum Registers
# ------------------------------------------

plain_qubits = QuantumRegister(16, 'P')
key_qubits = QuantumRegister(16, 'K')

# Measure only the unknown Low Byte
key_out = ClassicalRegister(8, 'c0')

# State register must come first because
# Row_Shift and MixColumns use absolute indices 0..15.

qc = QuantumCircuit(
    plain_qubits,
    key_qubits,
    key_out
)

# ------------------------------------------
# Initialize Plaintext
# ------------------------------------------

Plaintext(
    qc,
    plaintext,
    plain_qubits,
    length
)

# ------------------------------------------
# Initialize Key Superposition
# K = 0x00XX
#
# key_qubits[0:8]  = unknown Low Byte
# key_qubits[8:16] = fixed High Byte 0x00
# ------------------------------------------

Input_key(qc, key_qubits)

qc.barrier()

# ------------------------------------------
# Grover Search
# ------------------------------------------

# Search space = 2^8 = 256
# Assuming exactly one valid key:
# Optimal Grover iterations = 13

num_iterations = 1

for i in range(num_iterations):

    print(f"Iteration {i + 1}/{num_iterations}")

    # Oracle
    my_oracle(
        qc,
        plain_qubits,
        key_qubits,
        length,
        cipher_text
    )


    qc.barrier()

    # Diffuser acts ONLY on the unknown 8 key qubits
    qc.append(
        diffuser(8),
        key_qubits[:8]
    )

    qc.barrier()

# ------------------------------------------
# Measure only the unknown Low Byte
# ------------------------------------------

qc.measure(
    key_qubits[:8],
    key_out
)

# ------------------------------------------
# Circuit Information
# ------------------------------------------

print("Total qubits:", qc.num_qubits)
print("Classical bits:", qc.num_clbits)
print("Grover iterations:", num_iterations)
print("Circuit depth:", qc.depth())
print("Circuit size:", qc.size())


Iteration 1/1
Total qubits: 32
Classical bits: 8
Grover iterations: 1
Circuit depth: 190
Circuit size: 676


In [22]:

# ==========================================
# PART 4: SIMULATION AND RESULTS
# ==========================================

# ------------------------------------------
# Memory Before Execution
# ------------------------------------------

memory_before = get_memory_usage()

print(f"Memory before execution: {memory_before:.2f} MB")

# ------------------------------------------
# Initialize Simulator
# ------------------------------------------

backend = AerSimulator(
    method='matrix_product_state',
    precision='single'
)

sampler = SamplerV2(mode=backend)

# ------------------------------------------
# Transpile Circuit
# ------------------------------------------

qc_transpiled = transpile(
    qc,
    backend=backend,
    optimization_level=1
)

# ------------------------------------------
# Run Simulation
# ------------------------------------------

job = sampler.run(
    [qc_transpiled],
    shots=1024
)

result = job.result()

# ------------------------------------------
# Get Measurement Counts
# ------------------------------------------

counts_qc = result[0].data.c0.get_counts()

# Reverse bitstring display order
flipped_counts_qc = {
    bitstring[::-1]: value
    for bitstring, value in counts_qc.items()
}

print("\nOriginal counts:")
print(counts_qc)

print("\nFlipped counts:")
print(flipped_counts_qc)

# ------------------------------------------
# Plot Results
# ------------------------------------------

plot_histogram(flipped_counts_qc)

# ------------------------------------------
# Memory After Execution
# ------------------------------------------

memory_after = get_memory_usage()

print(f"Memory after execution: {memory_after:.2f} MB")

print(
    f"Memory used during execution: "
    f"{memory_after - memory_before:.2f} MB"
)


Memory before execution: 354.98 MB

Original counts:
{'10000000': 5, '00001000': 4, '01011010': 3, '01111000': 3, '00111000': 4, '11000010': 6, '11011001': 2, '11101010': 8, '01111001': 8, '10111001': 7, '00000100': 6, '00101000': 6, '10011001': 4, '00100100': 3, '10111100': 2, '10011100': 3, '11010101': 4, '01100110': 44, '11110100': 5, '00001111': 1, '11000000': 2, '11100111': 3, '00100000': 8, '01001011': 2, '01101001': 9, '01000110': 6, '00011000': 4, '01110001': 5, '01000000': 6, '10010111': 4, '11001111': 3, '01011001': 5, '01101111': 7, '00000110': 3, '00001001': 4, '10001000': 9, '11110110': 4, '10101101': 2, '00000111': 8, '11100100': 5, '00110010': 3, '10101111': 5, '11111101': 8, '10100001': 4, '10000100': 5, '00100110': 4, '10000111': 4, '01000010': 4, '10110001': 9, '00010011': 4, '11101000': 6, '10100110': 3, '01100010': 5, '11011101': 4, '10011011': 5, '10001111': 5, '10000011': 5, '11110011': 6, '01111100': 3, '00101111': 7, '10110101': 6, '00100010': 4, '00110100': 5, 

In [23]:

# ==========================================
# PART 5: BINARY TO HEXADECIMAL
# ==========================================

def binary_to_hex(binary_str):

    # Convert binary to hexadecimal
    hex_value = hex(int(binary_str, 2))[2:]

    # Ensure 2 hexadecimal digits for 8 bits
    return hex_value.upper().zfill(2)


# Convert all flipped binary results to hexadecimal
flipped_hex_counts = {
    binary_to_hex(bin_str): count
    for bin_str, count in flipped_counts_qc.items()
}


# Find the result with the highest measurement count
highest_count_key = max(
    flipped_counts_qc,
    key=flipped_counts_qc.get
)

highest_count_value = flipped_counts_qc[highest_count_key]


# Convert the most frequent result to hexadecimal
highest_hex = binary_to_hex(highest_count_key)


# ------------------------------------------
# Print results
# ------------------------------------------

print("Flipped counts (Binary):", flipped_counts_qc)

print("Flipped counts (Hex):", flipped_hex_counts)

print("Highest binary result:", highest_count_key)

print("Count of highest result:", highest_count_value)

print("Recovered Low Byte (Hex):", highest_hex)

print("Recovered Key (Hex):", "0x00" + highest_hex)


Flipped counts (Binary): {'00000001': 5, '00010000': 4, '01011010': 3, '00011110': 3, '00011100': 4, '01000011': 6, '10011011': 2, '01010111': 8, '10011110': 8, '10011101': 7, '00100000': 6, '00010100': 6, '10011001': 4, '00100100': 3, '00111101': 2, '00111001': 3, '10101011': 4, '01100110': 44, '00101111': 5, '11110000': 1, '00000011': 2, '11100111': 3, '00000100': 8, '11010010': 2, '10010110': 9, '01100010': 6, '00011000': 4, '10001110': 5, '00000010': 6, '11101001': 4, '11110011': 3, '10011010': 5, '11110110': 7, '01100000': 3, '10010000': 4, '00010001': 9, '01101111': 4, '10110101': 2, '11100000': 8, '00100111': 5, '01001100': 3, '11110101': 5, '10111111': 8, '10000101': 4, '00100001': 5, '01100100': 4, '11100001': 4, '01000010': 4, '10001101': 9, '11001000': 4, '00010111': 6, '01100101': 3, '01000110': 5, '10111011': 4, '11011001': 5, '11110001': 5, '11000001': 5, '11001111': 6, '00111110': 3, '11110100': 7, '10101101': 6, '01000100': 4, '00101100': 5, '10111000': 4, '00101001': 2

In [24]:

# ==========================================
# PART 6: ORIGINAL RESULTS (WITHOUT FLIPPING)
# ==========================================

def binary_to_hex(binary_str):

    hex_value = hex(int(binary_str, 2))[2:]

    return hex_value.upper().zfill(2)


# Convert original binary results to hexadecimal
hex_counts = {
    binary_to_hex(bin_str): count
    for bin_str, count in counts_qc.items()
}


# Find the highest-count result
highest_count_key = max(
    counts_qc,
    key=counts_qc.get
)

highest_count_value = counts_qc[highest_count_key]

# Convert highest result to hexadecimal
highest_hex = binary_to_hex(highest_count_key)


# Sort results by count and get top 3
top_three = sorted(
    counts_qc.items(),
    key=lambda x: x[1],
    reverse=True
)[:3]


# Convert top 3 results to hexadecimal
top_three_hex = [
    (binary_to_hex(bin_str), count)
    for bin_str, count in top_three
]


# ------------------------------------------
# Print results
# ------------------------------------------

print("Hex counts:", hex_counts)

print("Highest binary string:", highest_count_key)

print("Count of highest result:", highest_count_value)

print("Hexadecimal of highest result:", highest_hex)

print("Recovered Key:", "0x00" + highest_hex)


# Print top 3
print("\nTop three highest binary strings in hex:")

for hex_value, count in top_three_hex:

    print(
        f"Hex: {hex_value}, "
        f"Full Key: 0x00{hex_value}, "
        f"Count: {count}"
    )


Hex counts: {'80': 5, '08': 4, '5A': 3, '78': 3, '38': 4, 'C2': 6, 'D9': 2, 'EA': 8, '79': 8, 'B9': 7, '04': 6, '28': 6, '99': 4, '24': 3, 'BC': 2, '9C': 3, 'D5': 4, '66': 44, 'F4': 5, '0F': 1, 'C0': 2, 'E7': 3, '20': 8, '4B': 2, '69': 9, '46': 6, '18': 4, '71': 5, '40': 6, '97': 4, 'CF': 3, '59': 5, '6F': 7, '06': 3, '09': 4, '88': 9, 'F6': 4, 'AD': 2, '07': 8, 'E4': 5, '32': 3, 'AF': 5, 'FD': 8, 'A1': 4, '84': 5, '26': 4, '87': 4, '42': 4, 'B1': 9, '13': 4, 'E8': 6, 'A6': 3, '62': 5, 'DD': 4, '9B': 5, '8F': 5, '83': 5, 'F3': 6, '7C': 3, '2F': 7, 'B5': 6, '22': 4, '34': 5, '1D': 4, '94': 2, '17': 4, 'DC': 2, 'DB': 5, 'F2': 2, '12': 4, 'AC': 2, 'B0': 7, 'D3': 6, 'CA': 4, '2B': 8, '3E': 2, '93': 8, 'CE': 4, 'D6': 1, 'CD': 3, 'C3': 4, '21': 4, 'C9': 2, 'D4': 4, '49': 2, '9F': 4, '6B': 5, '67': 4, '82': 3, '0E': 1, '29': 3, '53': 1, '86': 4, '76': 5, '68': 3, 'AB': 4, '96': 3, '74': 4, 'F0': 6, '02': 7, '91': 3, '2C': 5, 'EF': 9, 'CB': 3, '65': 7, '0C': 3, 'F7': 7, '92': 5, 'A5': 3, '61':

In [25]:

# ==========================================
# PART 7: CIRCUIT ANALYSIS
# ==========================================

# Get the depth of the transpiled circuit
qc_depth = qc_transpiled.depth()

print("Transpiled Circuit Depth:", qc_depth)


# Get gate counts
gate_counts = qc_transpiled.count_ops()

print("\nGate Counts:")

for gate, count in gate_counts.items():
    print(f"{gate}: {count}")


# Total number of gates
total_gates = sum(gate_counts.values())

print("\nTotal Number of Gates:", total_gates)


# Number of qubits
print("Total Qubits:", qc_transpiled.num_qubits)


Transpiled Circuit Depth: 986

Gate Counts:
cx: 1176
p: 636
u2: 160
swap: 112
x: 58
ccx: 48
tdg: 40
t: 40
h: 32
barrier: 26
u3: 14
measure: 8

Total Number of Gates: 2350
Total Qubits: 32


In [26]:
# End execution timer
end_time = time.time()

# Calculate execution time
execution_time = end_time - start_time

# Calculate memory usage difference
memory_difference = memory_after - memory_before

print("\n===== Execution Performance =====")
print(f"Execution Time: {execution_time:.4f} seconds")

print(f"Memory Before Execution: {memory_before:.2f} MB")
print(f"Memory After Execution: {memory_after:.2f} MB")
print(f"Memory Difference: {memory_difference:.2f} MB")


===== Execution Performance =====
Execution Time: 3622.2248 seconds
Memory Before Execution: 354.98 MB
Memory After Execution: 356.89 MB
Memory Difference: 1.92 MB
